# 7 Bilingual PySpark: Blending Python and Sql Code

Diagram comparing logic structure between pyspark and sql
![pysparkVsSql.png](media/pysparkVsSql.png)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
import pyspark.sql.functions as F 
import pyspark.sql.types as T 
import logging

spark = SparkSession.builder.getOrCreate()
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

your 131072x1 screen size is bogus. expect trouble
25/01/05 22:06:25 WARN Utils: Your hostname, LAPTOP-CDHH1LA0 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/01/05 22:06:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/05 22:06:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# create dataframe
elements = spark.read.csv("data/elements/Periodic_Table_Of_Elements.csv", header=True, inferSchema=True)
elements.show()

25/01/05 22:06:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+----------+------+----------+----------------+---------------+-----------------+------+-----+-----+-----------+-------+-----+--------+---------+--------------------+------------+-----------------+---------------+-------+------------+------------+----------------+-------------------+----+------------+--------------+---------------+
|AtomicNumber|   Element|Symbol|AtomicMass|NumberofNeutrons|NumberofProtons|NumberofElectrons|Period|Group|Phase|Radioactive|Natural|Metal|Nonmetal|Metalloid|                Type|AtomicRadius|Electronegativity|FirstIonization|Density|MeltingPoint|BoilingPoint|NumberOfIsotopes|         Discoverer|Year|SpecificHeat|NumberofShells|NumberofValence|
+------------+----------+------+----------+----------------+---------------+-----------------+------+-----+-----+-----------+-------+-----+--------+---------+--------------------+------------+-----------------+---------------+-------+------------+------------+----------------+-------------------+----+----

In [3]:
elements.where(F.col("phase") == "liq").groupby("period").count().show()

+------+-----+
|period|count|
+------+-----+
|     6|    1|
|     4|    1|
+------+-----+



In [4]:
# to use sql to work with pyspark dataframes, one must register the dataframe as sql enabled with a method like createOrReplaceTempView() in the spark.catalog
# if you don't, you get an error like this example
try:
    spark.sql("""select period, count(*) from elements where phase='liq' group by period""").show()
except AnalysisException as e:
    logger.info(f"Expected error thrown: {e}", e)

--- Logging error ---
Traceback (most recent call last):
  File "/tmp/ipykernel_17906/2618172488.py", line 4, in <module>
    spark.sql("""select period, count(*) from elements where phase='liq' group by period""").show()
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hubert/data-analysis-pyspark/.venv/lib/python3.11/site-packages/pyspark/sql/session.py", line 1631, in sql
    return DataFrame(self._jsparkSession.sql(sqlQuery, litArgs), self)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hubert/data-analysis-pyspark/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "/home/hubert/data-analysis-pyspark/.venv/lib/python3.11/site-packages/pyspark/errors/exceptions/captured.py", line 185, in deco
    raise converted from None
pyspark.errors.exceptions.captured.AnalysisException: [TABLE_OR_V

In [5]:
# adding the element data frame to the spark catalog so it can be used via sql queries
"""
Here we are adding the elements dataframe to the spark catalog via `createOrReplaceTempView` which is one of a family of functions that include:
 - createTempView
 - createOrReplaceTempView
 - createGlobalTempView
 - createOrReplaceGlobalTempView

 The 'createOrReplace' prefix means that the function method will allow you to overwrite existing tempViews with the same name
 while the just `create` prefix means that the function method will throw an error if an overwrite is about to happen.
 
 A "global" temp view is one that is shared accross multiple spark sessions.

"""
elements.createOrReplaceTempView("elements")

try:
    spark.sql("""select period, count(*) from elements where phase='liq' group by period""").show()
except AnalysisException as e:
    logger.info(f"Expected error thrown: {e}", e)

+------+--------+
|period|count(1)|
+------+--------+
|     6|       1|
|     4|       1|
+------+--------+



In [6]:
"""
You can view and manage views in the spark catalog via `spark.catalog`
"""
# reset tables
elements.createOrReplaceTempView("elements")
# list views
tblList1 = spark.catalog.listTables()
assert len(tblList1) == 1
logger.info(f"table list: {tblList1}")

INFO:__main__:table list: [Table(name='elements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


In [7]:
# delete a view
spark.catalog.dropTempView("elements")
# verify that the view is deleted
tblListPostDrop = spark.catalog.listTables()
assert len(tblListPostDrop) == 0
logger.info(f"tbl list post drop: {tblListPostDrop}")

INFO:__main__:tbl list post drop: []


In [8]:
# download data for remainder of chapter

import requests
import shutil
import os 
import zipfile

def getAndSaveData(url: str, targetLoc: str):
    resp = requests.get(url, stream=True)
    targetLocParent = os.path.dirname(targetLoc)
    if not os.path.exists(targetLocParent):
        os.makedirs(targetLocParent)
    with open(targetLoc, 'wb') as out_file:
        shutil.copyfileobj(resp.raw, out_file)

def unzipData(targetZip: str, targetFolder: str):
    try:
        assert os.path.exists(targetZip)
    except AssertionError as ae:
        logger.error(f"Unable to locate target zip at: {targetZip}")
    if not os.path.exists(targetFolder):
        os.makedirs(targetFolder)
    with zipfile.ZipFile(targetZip, 'r') as zip_ref:
        zip_ref.extractall(targetFolder)

In [9]:
data_dir = "./data/backblaze"
q32019ZipTgt = os.path.join(data_dir, "data_Q3_2019.zip")
q32019ZipSrc = "https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/data_Q3_2019.zip"
q32019UnzipTgt = os.path.join(data_dir, "data_Q3_2019")

In [10]:
if (not os.path.exists(q32019ZipTgt)) and not (os.path.exists(q32019UnzipTgt)):
    getAndSaveData(q32019ZipSrc, q32019ZipTgt)
if not (os.path.exists(q32019UnzipTgt)):
    unzipData(q32019ZipTgt, q32019UnzipTgt)

In [11]:
# collate data into a single dataframe
q1 = None
q1Path = os.path.join(data_dir, "drive_stats_2019_Q1")
if os.path.exists(q1Path):
    q1 = spark.read.csv(q1Path, header=True, inferSchema=True)
q2 = None 
q2Path = os.path.join(data_dir, "data_Q2_2019")
if os.path.exists(q2Path):
    q2 = spark.read.csv(q2Path, header=True, inferSchema=True)
q3 = None 
q3Path = q32019UnzipTgt
if os.path.exists(q3Path):
    q3 = spark.read.csv(q3Path, header=True, inferSchema=True)
q4 = None 
q4Path = os.path.join(data_dir, "data_Q4_2019")
if os.path.exists(q4Path):
    q4 = spark.read.csv(q4Path, header=True, inferSchema=True)

backblaze_2019 = q3
if not q4 == None:
    q4_fields_extra = set(q4.columns) - set(q1.columns)
    for i in q4_fields_extra:
        q1 = q1.withColumn(i, F.lit(None).cast(T.StringType()))
        q2 = q2.withColumn(i, F.lit(None).cast(T.StringType()))
        q3 = q3.withColumn(i, F.lit(None).cast(T.StringType()))
    backblaze_2019 = (
        q1.select(q4.columns)
        .union(q2.select(q4.columns))
        .union(q3.select(q4.columns))
        .union(q4)
    )

backblaze_2019 = backblaze_2019.select(
    [
        F.col(x).cast(T.LongType()) if x.startswith("smart") else F.col(x)
        for x in backblaze_2019.columns
    ]
)
backblaze_2019.createOrReplaceTempView("backblaze_stats_2019")
# qds = [q1, q2, q3, q4]
# for qd in qds:
#     if not None == qd:
#         for in in


In [12]:
# comparing select (column filtering) and where (row filtering)
spark.sql("select serial_number from backblaze_stats_2019 where failure = 1").show(5)

backblaze_2019.where("failure = 1").select("serial_number").show(5)

+-------------+
|serial_number|
+-------------+
|     ZA10MCJ5|
|     ZCH07T9K|
|     ZCH0CA7Z|
|     Z302F381|
|     ZCH0B3Z2|
+-------------+
only showing top 5 rows

+-------------+
|serial_number|
+-------------+
|     ZA10MCJ5|
|     ZCH07T9K|
|     ZCH0CA7Z|
|     Z302F381|
|     ZCH0B3Z2|
+-------------+
only showing top 5 rows



In [13]:
spark.sql(
    """select model,
    min(capacity_bytes/ pow(1024,3)) min_GB,
    max(capacity_bytes/ pow(1024,3)) max_GB
    FROM backblaze_stats_2019
    GROUP BY 1
    ORDER BY 3 DESC
    """
).show()

+--------------------+--------------------+-----------------+
|               model|              min_GB|           max_GB|
+--------------------+--------------------+-----------------+
| TOSHIBA MG07ACA14TA|             13039.0|          13039.0|
|       ST12000NM0007|-9.31322574615478...|          11176.0|
|       ST12000NM0117|             11176.0|          11176.0|
|HGST HUH721212ALN604|-9.31322574615478...|          11176.0|
|HGST HUH721212ALE600|             11176.0|          11176.0|
|       ST10000NM0086|-9.31322574615478...|           9314.0|
|HGST HUH721010ALE600|-9.31322574615478...|           9314.0|
|         ST8000DM002|-9.31322574615478...|7452.036460876465|
|     TOSHIBA HDWF180|   7452.036460876465|7452.036460876465|
|         ST8000DM005|   7452.036460876465|7452.036460876465|
|        ST8000NM0055|-9.31322574615478...|7452.036460876465|
|HGST HUH728080ALE600|-9.31322574615478...|7452.036460876465|
|         ST8000DM004|   7452.036460876465|7452.036460876465|
|       